Please run the above cell. If it indicates success, you might need to run the full setup cell again or restart your Colab runtime to fully apply the changes and re-initialize the TTS model correctly.

In [ ]:
# --- 1. CLEAN WORKSPACE & SYSTEM PACKAGES ---
!apt-get update && apt-get install -y espeak-ng libsndfile1
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py
!rm -rf TTS Trainer

# --- 2. CLONE SOURCE CODE ---
!git clone --depth 1 --branch v0.22.0 https://github.com/coqui-ai/TTS.git
!git clone --depth 1 https://github.com/coqui-ai/Trainer.git

# --- 3. PATH INJECTION (Bypass Python 3.12 restrictions) ---
import sys
import os

paths = [os.path.abspath("TTS"), os.path.abspath("Trainer")]
for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)
print(" Paths mapped successfully.")

# --- 4. INSTALL ALL DEPENDENCIES (Swapped pyngrok for pycloudflared) ---
!pip install coqpit==0.0.17 fastapi uvicorn pydub transformers==4.48.0 \
            num2words pysbd librosa mecab-python3 unidic-lite anyio \
            starlette pydantic numba scikit-learn soundfile pooch soxr \
            lazy_loader msgpack gruut jieba cython tensorboard \
            anyascii pypinyin g2p_en bangla bnnumerizer einops \
            bnunicodenormalizer jamo hangul-romanize sudachipy sudachidict_core \
            pycloudflared

# --- 5. THE DOT-DICT PATCH (Fixes Python 3.12 Dict Error) ---
import typing
import torch
import coqpit.coqpit
class DotDict(dict):
    def __getattr__(self, name):
        try:
            val = self[name]
            if isinstance(val, dict) and not isinstance(val, DotDict):
                val = DotDict(val)
                self[name] = val
            elif isinstance(val, list):
                val = [DotDict(v) if isinstance(v, dict) else v for v in val]
                self[name] = val
            return val
        except KeyError:
            raise AttributeError(f"Config dictionary has no attribute '{name}'")
    def __setattr__(self, name, value):
        self[name] = value

_orig_deserialize = coqpit.coqpit._deserialize
def aggressive_deserialize(x, field_type):
    if isinstance(x, dict): x = DotDict(x)
    try:
        res = _orig_deserialize(x, field_type)
        if isinstance(res, dict) and not isinstance(res, DotDict):
            return DotDict(res)
        return res
    except Exception:
        return x
coqpit.coqpit._deserialize = aggressive_deserialize

# --- 6. THE PYTORCH 2.6 PATCH (Fixes the UnpicklingError) ---
_original_load = torch.load
def legacy_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = legacy_load

# --- 7. INITIALIZE XTTS MODEL ---
from TTS.api import TTS
os.environ["COQUI_TOS_AGREED"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n ALL MODULES LOADED! Downloading XTTS v2 to {device}...")
tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("\n🎉 MODEL READY AND LOADED!")

# --- 8. FASTAPI SERVER ---
from fastapi import FastAPI, Request, UploadFile, File
from fastapi.responses import FileResponse
import threading
import uvicorn

app = FastAPI()

@app.post("/upload")
async def upload(file: UploadFile = File(...)):
    os.makedirs("/content/speakers", exist_ok=True)
    with open("/content/speakers/voice.wav", "wb") as f: f.write(await file.read())
    return {"status": "Uploaded"}

@app.post("/generate")
async def generate(request: Request):
    data = await request.json()
    tts.tts_to_file(
        text=data['text'],
        speaker_wav="/content/speakers/voice.wav",
        language="en",
        file_path="/content/output.wav"
    )
    return FileResponse("/content/output.wav")

# --- 9. START CLOUDFLARE TUNNEL (Bypasses Ngrok Outage) ---
from pycloudflared import try_cloudflare

# Start the FastAPI server in the background
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()

# Open the Cloudflare Tunnel
try:
    tunnel = try_cloudflare(port=8000)
    print(f"\n API IS LIVE! COPY THIS URL TO VS CODE: {tunnel.tunnel}\n")
except Exception as e:
    print(f"\n Tunnel info: {e}.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,840 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,118 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-d

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")



✅ ALL MODULES LOADED! Downloading XTTS v2 to cuda...
 > Downloading model to /root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2


100%|█████████▉| 1.86G/1.87G [00:27<00:00, 63.5MiB/s]
100%|██████████| 1.87G/1.87G [00:27<00:00, 67.2MiB/s]
4.37kiB [00:00, 17.6kiB/s]

361kiB [00:00, 1.46MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 65.8iB/s]
  0%|          | 1.02k/7.75M [00:00<23:49, 5.42kiB/s]

 > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.


...

GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.



🎉 MODEL READY AND LOADED!


INFO:     Started server process [1866]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Download cloudflared...:   0%|          | 0/39261733 [00:00<?, ?it/s]

 * Running on https://busy-restrict-rider-manual.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics

🚀 API IS LIVE! COPY THIS URL TO VS CODE: https://busy-restrict-rider-manual.trycloudflare.com

